# **Recurrent Language Modeling**
<img src="https://static01.nyt.com/images/2018/05/15/arts/01hal-voice1/merlin_135847308_098289a6-90ee-461b-88e2-20920469f96a-superJumbo.jpg" width=20% style="border-radius:20px"><br>
Let's teach computer to speak... (cause what can go wrong?)

## Table Of contents
- What is this notebook about?
- What are RNNs?
- Character-level RNN LM Implementation
- What are LSTMs and GRUs?
- Character-level LSTM and GRU LMs Implementation
- Language Modeling fun (cause why not?)
- Word Level language Modeling
- Word embeddings
- Word-level RNN LM Implementation
- Word-level LSTM and GRU LMs Implementation
- More fun!
- **Tiny Stories** and Finita La Comedia!

### What is this notebook about?
This is an introductory-level guide to language modeling with RNNs and LSTMs. and GRUs!<br>
In this one my friends we will not only discuss all the details of Recurrent Language Modeling but we will also apply it to interesting tasks, such as:
- Novel writing
- Programming (oh no... AGI takes my job!)
- LaTex
- Story Extending (Balabebe)

**Sounds Interesting? Good!**

### What are RNNs?
RNN - Recurrent Neural Network - special framework (type of neural networks), which processes input sequentially sharing all the parameters.<br>
It applies the same function to each sequence token and carries information about the past in special **hidden state**.<br>
There are various RNN visualizations, but find all of the quite confusing, because they either express nothing or give wrong ideas about practical implementation.<br>
On the image below you see a visualization of RNN wrapped out in time.<br>
<img src="https://i.imgur.com/Iveichs.jpg" style="border-radius: 20px" width=30%><br>
As you can see on each timestep RNN (green block) processes an input (character in this case) and a hidden state (it comes from left to right through the time).<br>
On each time step RNN returns an output token (at timestep t=1 it recieves input "h" and returns output "e". On the scheme visually it goes up) and updates hidden state (this one goes to the right).<br>
On intuitive level it learns to predict the next token given input token and using the previous knowledge stored in hidden state.<br>
**Important technical note!**
RNN is not a set of individual blocks, it is a one Module, which is called multiple times accumulating gradients.<br>
In other words, on practice it's more like this:<br>
<img src="https://colah.github.io/posts/2015-08-Understanding-LSTMs/img/RNN-rolled.png" width=10%><br>
But this scheme explaines nothing at all.<br>

**Important sanity note!**
RNNs are not only used for Language Modeling. They can play a role of an encoder for classification/regression tasks, they are used to work with time-series data. They are capable of more things!

But what about backpropagation? How is it possible?<br>
Well, it's called **"Backpropagation through time"** and for a good reason.<br>
At each timestep we calculate loss. We take average loss over timesteps and backpropagate from it.<br>
<img src="https://wikidocs.net/images/page/160068/7_Backpropagation-in-RNNs.jpg" style="border-radius:20px" width=30%><br>

### Character-level RNN LM Implementation

In [2]:
import torch
from torch import nn, optim
import torch.nn.functional as F
from tqdm import tqdm

import numpy as np

In [3]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [4]:
# Path is "../../Data/NLP/onegin.txt"
def load_preprocess(path, chunk_size=256, batch_size=8):
    with open(path, "r") as f:
        data = f.read()
    
    vocab = sorted(set(data)) + ["<", ">"]
    vocab_size = len(vocab)

    itos = {i:s for i, s in enumerate(vocab)}
    stoi = {s:i for i, s in itos.items()}

    chunks = [data[i:i+chunk_size] for i in range(0, len(data)-chunk_size, chunk_size)]
    chunks = ["<" + c + ">" for c in chunks]
    chunks = [[stoi[ch] for ch in chunk] for chunk in chunks]
    input = [ch[:-1] for ch in chunks]
    target = [torch.tensor(ch[1:], dtype=torch.long) for ch in chunks]
    input = [F.one_hot(torch.tensor(inp), vocab_size).float() for inp in input]

    inp_batches = [input[i:i+batch_size] for i in range(0, len(input)-batch_size, batch_size)]
    trg_batches = [target[i:i+batch_size] for i in range(0, len(target)-batch_size, batch_size)]

    inp_matrix = torch.stack([torch.stack(batch, dim=0) for batch in inp_batches], dim=0)
    trg_matrix = torch.stack([torch.stack(batch, dim=0) for batch in trg_batches], dim=0)

    return inp_matrix, trg_matrix, vocab, vocab_size, itos, stoi

In [37]:
class CharacterRNN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, batch_size=8):
        super().__init__()
        self.hidden_size = hidden_size
        self.batch_size = batch_size

        self.input_to_hidden = nn.Linear(input_size + hidden_size, hidden_size)
        self.activation = nn.Tanh()
        self.hidden_to_output = nn.Linear(hidden_size, output_size)
    
    def init_hidden(self, device, inference=False):
        if inference:
            return torch.zeros(1, self.hidden_size, device=device)
        else:
            return torch.zeros(self.batch_size, self.hidden_size, device=device)

    def forward(self, input, hidden):
        # Input shape: (8, 147)
        # Hidden shape: (8, hidden_size)
        # Concat along -1: (8, 147+hidden_size)
        input_with_hidden = torch.cat([input, hidden], dim=-1)
        new_hidden = self.activation(self.input_to_hidden(input_with_hidden))
        output = self.hidden_to_output(new_hidden)
        return output, new_hidden

In [38]:
from tqdm import tqdm

In [39]:
def generate(model, starter, stoi, itos, vocab_size):
    sequence = starter

    with torch.inference_mode():
        hidden = model.init_hidden(device, inference=True)
        while True:
            input = F.one_hot(torch.tensor([stoi[sequence[-1]]]), vocab_size).float().to(device)
            output, hidden = model(input, hidden)
            output_char = itos[torch.multinomial(torch.softmax(output, dim=1), 1).item()]
            if output_char == ">":
                break
            sequence += output_char
    return sequence

In [49]:
def train(model, criterion, optimizer, file_path, chunk_size=256, batch_size=8, epochs=20):
    model.batch_size = batch_size
    input, target, vocab, vocab_size, itos, stoi = load_preprocess(file_path, chunk_size=chunk_size, batch_size=batch_size)
    for epoch in tqdm(range(epochs)):
        for input_batch, target_batch in zip(input, target):
            input_batch = input_batch.to(device)
            target_batch = target_batch.to(device)
            
            loss = 0
            hiddens = model.init_hidden(device)
            for t in range(input_batch.shape[1]):
                character_t = input_batch[:, t, :]
                target_t = target_batch[:, t]

                predictions_t, hiddens = model(character_t, hiddens)
                loss_t = criterion(predictions_t, target_t)
                loss += loss_t
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        if ((epoch+1) % 10) == 0:
            print(f"Epoch: {epoch+1} | Loss: {loss.item()}")
            print(generate(model, "<", stoi, itos, vocab_size))
    
    return model, stoi, itos, vocab_size

In [50]:
path = "../../Data/NLP/onegin.txt"
model = CharacterRNN(147, 512, 147).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
model, stoi, itos, vocab_size = train(model, criterion, optimizer, path, epochs=100, chunk_size=128, batch_size=16)

 10%|█         | 10/100 [00:33<05:02,  3.37s/it]

Epoch: 10 | Loss: 276.6661682128906
<, Хмие варь у у соми ; cаз,
Блю подно койсне)сы,
Замолленяя 


 20%|██        | 20/100 [01:07<04:30,  3.39s/it]

Epoch: 20 | Loss: 257.81787109375
<лим,
Что пезвой,
Гредне, подой на буль.
Как фиены мы одалит
Он про дебрисн


 30%|███       | 30/100 [01:41<03:55,  3.36s/it]

...

Epoch: 80 | Loss: 182.95883178710938
<и нет:
Онегин тограни сам инсягданнон (згаского любовния большей,
В свое чуманиты разрозной
Его лено брокаят уны,
Доба моегонко васничароз карано: Ититияженою ила Мерови
И. кобуднадною нагредол... хлани кля долог обкрывыло...
(у очаровадовых


 90%|█████████ | 90/100 [05:03<00:33,  3.38s/it]

Epoch: 90 | Loss: 215.37828063964844
<чувства в тумно завсталь,
Воро кикой восшем усичныхисмивмусто в семейса: вычалов,
126
Ватила Лень; ручью м.и. воснольно сьмохва!
Быва разгкагу, что чем он чистительным повости,
Когда в уда герная предокным,


100%|██████████| 100/100 [05:36<00:00,  3.37s/it]

Epoch: 100 | Loss: 228.40245056152344
< зналиет, плахил силгие, и
. . . . . .


In [56]:
print(generate(model, "О", stoi, itos, vocab_size))

Онегин вышертым стеховски,
Ивил емет и постраст обуль
Пригнеж не миражей портрусты;
Превеждовлицу предрого, дан,
Хоть и рашенык, в тончен он.
А я! радом в вошило,
В голявои темновой видок демитилься, боремы
Месною грах, в вдолиды деть мы в них готов.
XLII

хожит Онегинем живных,
Завых лицы все провь;
Когорышью: я ожан


In [57]:
class CharacterDeepRNN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, batch_size=16):
        super().__init__()
        self.hidden_size = hidden_size
        self.batch_size = batch_size

        self.input_to_hidden1 = nn.Linear(input_size + hidden_size, hidden_size)
        self.activation1 = nn.ReLU()
        self.hidden1_to_hidden2 = nn.Linear(hidden_size * 2, hidden_size)
        self.activation2 = nn.ReLU()
        self.hidden2_to_hidden3 = nn.Linear(hidden_size * 2, hidden_size)
        self.activation3 = nn.ReLU()
        self.hidden3_to_output = nn.Linear(hidden_size, output_size)
    
    def init_hidden(self, device, inference=False):
        if inference:
            return [torch.zeros(1, self.hidden_size, device=device) for _ in range(3)]
        else:
            return [torch.zeros(self.batch_size, self.hidden_size, device=device) for _ in range(3)]

    def forward(self, input, hiddens):
        h1, h2, h3 = hiddens

        input_with_hidden1 = torch.cat([input, h1], dim=-1)
        new_hidden1 = self.activation1(self.input_to_hidden1(input_with_hidden1))

        hidden1_with_hidden2 = torch.cat([new_hidden1, h2], dim=-1)
        new_hidden2 = self.activation2(self.hidden1_to_hidden2(hidden1_with_hidden2))

        hidden2_with_hidden3 = torch.cat([new_hidden2, h3], dim=-1)
        new_hidden3 = self.activation3(self.hidden2_to_hidden3(hidden2_with_hidden3))

        output = self.hidden3_to_output(new_hidden3)
        return output, [new_hidden1, new_hidden2, new_hidden3]

In [58]:
path = "../../Data/NLP/onegin.txt"
model1 = CharacterDeepRNN(147, 512, 147).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model1.parameters(), lr=0.001)
model1, stoi, itos, vocab_size = train(model1, criterion, optimizer, path, epochs=100, chunk_size=256, batch_size=16)

 10%|█         | 10/100 [01:01<09:12,  6.14s/it]

Epoch: 10 | Loss: 620.4088134765625
< 4ж быр вумдревьсно
И дачай
С реэт гая Телминких с зек шимера Бигу, ропзу я ьна) Гюлой онит, ны оз в варянь, инавея истьвю на ит Леччы-брей, 


 20%|██        | 20/100 [02:02<08:08,  6.10s/it]

Epoch: 20 | Loss: 575.9990844726562
< Л тубра.
Гряя впехо мошлампе размрый остатыми Ускари всё вож дюхают лака»
— Ах латон


 30%|███       | 30/100 [03:03<07:06,  6.09s/it]

...

Epoch: 80 | Loss: 500.3745422363281
<аме?
Беркали тахарек слова бых собринил ее шогдах.
Онегин мой сколя улицею,
Чеклим на ручипой украхою трори вышерка!
Егр умез подрузей у чества измечит, и Симерков ознатойхоя взорое). Накайа,
Мельях риаброминие нероды милой Вонней


 90%|█████████ | 90/100 [09:09<01:01,  6.10s/it]

Epoch: 90 | Loss: 482.8556823730469
<
1. . . . . . . 
. . . . . . . . IX1

Что ждищив стрещиг тривищив;
127
«У Идрый свою ли любили
Це партиным прилежил нах не пословстивица!
На ды-кнодился мою
М


100%|██████████| 100/100 [10:09<00:00,  6.10s/it]

Epoch: 100 | Loss: 474.8013000488281
<ругама шулебем
Ровинчествы брогиков просторыем строкница
(С гочиною вобратьяных,
Забутие душникся завриду.
Он пирепнык чиньте даму, 


In [65]:
print(generate(model1, "О", stoi, itos, vocab_size))

Он порачестечную дло счепил Перыдение была
Поэтой? как, оквершанные достроеблива...»
— Стряшиц этею молл без ходмал, Аемгеряй мгон?..»
XXIV

Евгений здут седомною посшен...
Уны, забронечных врости,
И вмем вы проходался крапетднаемный прич нашихого мюсиной.)
XI

Татьяна седу приволечке вы прогжуле смердовая;
Грусто сарых перноки котери,
. . . . . . . . . . . . . . . . . .
III

Как успявшив кагвах ноя уже;
В так вас их мия счустинские лирковы7 прохрасть». — Ниют, кревонсевною верелевок сврозавци


In [66]:
class CharacterLSTM(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, batch_size=16):
        super().__init__()
        self.hidden_size = hidden_size
        self.batch_size = batch_size

        self.f_t = nn.Linear(input_size + hidden_size, hidden_size)
        self.f_t_gate = nn.Sigmoid()

        self.i_t = nn.Linear(input_size + hidden_size, hidden_size)
        self.i_t_gate = nn.Sigmoid()
        self._c_t = nn.Linear(input_size + hidden_size, hidden_size)
        self._c_t_act = nn.Tanh()

        self.o_t = nn.Linear(input_size + hidden_size, hidden_size)
        self.o_t_gate = nn.Sigmoid()
        self.c_t_act = nn.Tanh()

        self.output = nn.Linear(hidden_size, output_size)
    
    # Actually, should've better called it init_states, but I'm
    # Constrained with training function name assumptions.
    def init_hidden(self, device, inference=False):
        if inference:
            return [torch.zeros(1, self.hidden_size, device=device) for _ in range(2)]
        else:
            return [torch.zeros(self.batch_size, self.hidden_size, device=device) for _ in range(2)]

    def forward(self, input, states):
        hid, cell = states

        input_with_hidden = torch.cat([input, hid], dim=-1)

        f_t = self.f_t_gate(self.f_t(input_with_hidden))

        i_t = self.i_t_gate(self.i_t(input_with_hidden))
        _c_t = self._c_t_act(self._c_t(input_with_hidden))

        updated_cell = f_t * cell + i_t * _c_t

        o_t = self.o_t_gate(self.o_t(input_with_hidden))
        new_hidden = o_t * self.c_t_act(updated_cell)

        output = self.output(new_hidden)

        return output, [new_hidden, updated_cell]


In [68]:
path = "../../Data/NLP/onegin.txt"
model2 = CharacterLSTM(147, 512, 147).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model2.parameters(), lr=0.001)
model2, stoi, itos, vocab_size = train(model2, criterion, optimizer, path, epochs=100, chunk_size=512, batch_size=32)

 10%|█         | 10/100 [00:39<05:52,  3.92s/it]

Epoch: 10 | Loss: 1745.379150390625
<


 19%|█▉        | 19/100 [01:14<05:17,  3.92s/it]

Epoch: 20 | Loss: 1471.9879150390625


 20%|██        | 20/100 [01:19<05:30,  4.13s/it]

...

Epoch: 80 | Loss: 1196.4583740234375
< лпочыхат ковы
Божертой мазвудет


 90%|█████████ | 90/100 [05:53<00:38,  3.87s/it]

Epoch: 90 | Loss: 1169.9598388671875
<Y краго былечный прою живсечушьск


100%|██████████| 100/100 [06:32<00:00,  3.92s/it]

Epoch: 100 | Loss: 1148.3079833984375
<, мыши
Он оменни.сни потреен им
22 сuf Nilenr. До


In [109]:
print(generate(model2, "О", stoi, itos, vocab_size))

Онай мочной голаной
Я вледних мог;учтивы, друбать?
Здерьян порсать Там зобровстеты;
Сприслихлой, зя ребод сечдамы
Вередластий лок прустыму. К негу.
Поит две чепистует пени;
Ктго, вда толя? — отв везгрунит,
Касся таль кле к жирчно преблова,
К обружилво киека в дет;
Проко е пуловат и умен,
От пузвую соной Еюботь.
XV

Тотьяти зе милой довечнык
Рапруть на, ловшемных поче,
Не вокрадансекий вое другик
Нах не бухашособ онов,
Дардуятья, ветсе Татевся.
XLII

Она нас багда бла назети стрева!
Та гоуз бель обего кожня,
И прка шасть не сястро нна 
Шей, кругы, праклов и ствах
Еще ужаликой ист.
. . . . . . . . . . . . . . . . . . . 
 тва на вад, поче;
Пу пол го гомый, глете стелой,
Говорорчкохит седа ребя;
Те пркия, ледьем об лестной!
Зодаят паза нахожень
Кредновоквонока пелашей,
На тоетесь я то чазкиладцо.
И у тас олугказ ежда:
Ол го дврога уго лан,
Той иходинае точенье
Презмими чутвым бе енепара
Ди машв? Тамною одше ск.
Соторвет


In [143]:
class CharacterGRU(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, batch_size=16):
        super().__init__()
        self.hidden_size = hidden_size
        self.batch_size = batch_size

        self.r_t = nn.Linear(input_size + hidden_size, hidden_size)
        self.r_t_gate = nn.Sigmoid()
        self.z_t = nn.Linear(input_size + hidden_size, hidden_size)
        self.z_t_gate = nn.Sigmoid()
        self._h_t = nn.Linear(input_size + hidden_size, hidden_size)
        self._h_t_act = nn.Tanh()
        self.out = nn.Linear(hidden_size, output_size)
    
    
    def init_hidden(self, device, inference=False):
        if inference:
            return torch.zeros(1, self.hidden_size, device=device)
        else:
            return torch.zeros(self.batch_size, self.hidden_size, device=device)

    def forward(self, input, hidden):
        input_with_hidden = torch.cat([input, hidden], dim=-1)
        z_t = self.z_t_gate(self.z_t(input_with_hidden))
        r_t = self.r_t_gate(self.r_t(input_with_hidden))
        
        hid_r_t = hidden * r_t

        candidate_input = torch.cat([input, hid_r_t], dim=-1)
        _h_t = self._h_t_act(self._h_t(candidate_input))
        new_hidden = (1 - z_t) * hidden + z_t * _h_t
        output = self.out(new_hidden)

        return output, new_hidden


In [145]:
path = "../../Data/NLP/onegin.txt"
model3 = CharacterGRU(147, 1024, 147, batch_size=16).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model3.parameters(), lr=0.001)
model3, stoi, itos, vocab_size = train(model3, criterion, optimizer, path, epochs=250, chunk_size=512, batch_size=32)

  4%|▍         | 10/250 [00:37<14:55,  3.73s/it]

Epoch: 10 | Loss: 1562.0732421875
<пухкгчучуббмняв т тнойСвел,ьоараросе нерогн плс,зСэ л всд дой,ечстечоуды
аjнкд
вомачи зцестор;сованпшб


  8%|▊         | 20/250 [01:14<14:22,  3.75s/it]

Epoch: 20 | Loss: 1323.5631103515625
<),
И слета,ья стож. — паты.,
Не постим ску, мнай и пврешег на.а3
... т й бом ваут!
Кад лавщесст у стружиж одой
В л грибя поз брозруде сталь серяд, кы
ойтрво нелен
 раком перя идая иЛ ный!У: т е прv3к
. чель
ка лико влике пыгужирас
Погомежбехдлер си екижетьох долек
Vо вумь,гож о гл оатой.л.й.
rт  либли туй прого:
Л ких дав  рддею в вошмром,...,
И срубок лсрамуа вестжет.,Спегие ньгонецьо  оправ свохи
tк вигна праскиу в зловмыл, Она ним бы снорина. Ены
АseXII

Срамаглабяв счсего3.Псхлеви ний скоэт:
Перею,кл в погат ич пнош жемебиа
XXXII

«ужсечне бесвках сотлади ка ссно


 12%|█▏        | 30/250 [01:51<13:35,  3.71s/it]

...

Epoch: 230 | Loss: 9.283625602722168
<
Monsieun 1) Илашиц новреден ей щушит: «Дерога земетра дго тенжул но Sti a breny, oun doure фогуре—
Но руко нагосу ногох;
но ското изнала без суднуй,
Симу какать в праздлук моиных,
Вогоняй яхалюн я нем вечной,
С млачум щетел, и крысь титьет,
Умы пушаятного звине.
Она ило пуши до угала,
А плоповал суж бе Егое
И быде замеща их тум
Ота неждо ж


 96%|█████████▌| 240/250 [14:47<00:37,  3.74s/it]

Epoch: 240 | Loss: 8.798795700073242
<тронут был:
Язык девических мечтаний
В нем думы роем возмутил;
И вспомнил он Татьяны милой
И бледный цвет и вид унылый;
И в сладостный, безгрешный сон
Душою погрузился он.
XVIII

Когда б ан бруге зистов двятся должавай
Ужали с сробо полудашном
Наорающий ракоютый
Подлупнет скустью утолоны:
Как тся, втобнов наж портый

Сердкамисьядцем приснаюсла;
Орны привычка удренья,
Семянных призупорон,
И скаду симый вах умей,
И я беснет она причодна.
XLV

И чурство вешный при лене,
Не хадет натринница двях
И ночни передо умнее
Стренятся в рн


100%|██████████| 250/250 [15:24<00:00,  3.70s/it]

Epoch: 250 | Loss: 8.572463989257812
<
XIX

Бля остовесна отехлава
Стоет топор беж обкалать:
Поклалося нашей шугдет
Тетвя богружднним стещай
Безутрым нет; Фриня довглед,
Но всё тоОнется поэтой.
Она геряяк паткноког
В серецкаха поррггали
На к жизцифенье своей.
1) 
О тре любяль ещет дустол;
Кугка в идетраный гразит,
В ней нам Фиатьной паменны.
XLI

На ве, иславлявенной дврога
Я встроплеть в схоти в обма.
На эте все плящетн


In [183]:
print(generate(model3, "О", stoi, itos, vocab_size))

Они ил латав;
Не тах любять ват вышей балыш:
Тен, Бовлод: ты, мчи гисний,
Прилазым рнеге; разъевор,
Оню ипалена пройред...
И нак не чадном спесе дать;
Беред замитенькие висла
Вымока рениченьем взорм.
XXXII

Кта ж еще в ташно. Порицо.
В согда сердвем и не мог

Сю лице, ранесь, пошол
Вадирее пратит зередом,
Которой в своег в терной он
Илы поэти роковой;
Дря сколь радит: То я сказал!
XXII

Я зо лтая в неги свой своей
Был жества с обраною рекой.
Ж ваняньею: вечны кокры,
Но чеот сердцк юдним Убиды,
И нежки, тажной и прижне,
Где талко тание линь,
Отегон всем нарквет гидой,

